In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import torch
import copy
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from utils import remove_final_activation, plot_image_with_label, gradcam_on_input, show_gradcam_on_image
from dataset import BarrettsHRMEDataset, create_dataloader

In [5]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [6]:
model = torch.load("./project_dir/trained_models/resnet/models/model_epoch10.pt", weights_only=False)
model = model.to(device)

In [7]:
annotations_csv_filepath = "./HRME_image_annotations_dataset.csv"
df = pd.read_csv(annotations_csv_filepath)

In [8]:
test_dataset = BarrettsHRMEDataset(annotations_csv_filepath, data_partition="test")
test_dataloader = create_dataloader(test_dataset, batch_size=1)

In [9]:
# Visualize GradCAM on misclassified images
cam_model = copy.deepcopy(model)
cam_model = remove_final_activation(cam_model)
target_layer = cam_model.layer4[-1]
cam_model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
for input_tensor, labels, samples in test_dataloader:
    input_tensor = input_tensor.to(device)
    grayscale_cam, model_outputs = gradcam_on_input(cam_model, input_tensor, [target_layer])
    for i in range(len(samples)):
        original_img = cv2.imread(samples[i]['image_filepath'], cv2.IMREAD_UNCHANGED)
        plt.imshow(original_img, cmap='gray')
        plt.axis('off')
        plt.show()
        cam_image = show_gradcam_on_image(samples[i], grayscale_cam[i], alpha=0.15)
        plot_image_with_label(cam_image, f"Pred: {model_outputs[i]:.3f}, True: {labels[i]}")
        plt.show()
        break
    break

In [14]:
grayscale_cam.dtype

dtype('float32')

In [ ]:

# Create a dummy 2D array that spans 0 to 1
gradient = np.linspace(0, 1, 256).reshape(1, -1)

# Show it with jet colormap
plt.imshow(gradient, aspect='auto', cmap='jet')
plt.gca().set_visible(False)  # Hide the axes
plt.colorbar(orientation='horizontal', label='Activation')
plt.show()